In [1]:
import os 
from dotenv import load_dotenv
from openai import OpenAI
import json
from IPython.display import display, Markdown


In [2]:
# Load environment variables from .env file
load_dotenv()
api_key =   os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment variables. Please set it in your .env file.")
else:
    print("API key loaded successfully.")


API key loaded successfully.


In [3]:
MODEL = "gpt-5-nano"
openai = OpenAI(api_key=api_key)

In [4]:
response = openai.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant that generates sales brochures for products."
        },
        {
            "role": "user",
            "content": "Generate a sales brochure for a new smartphone called 'TechPhone X' with the following features: 1. 6.5-inch OLED display, 2. Triple-lens camera system, 3. 5G connectivity, 4. Long-lasting battery life, 5. Sleek design."
        }
    ]
)
response.choices[0].message.content

'TechPhone X: Redefine Your Everyday\n\nA smartphone that pairs breathtaking visuals with pro-grade photography, blazing-fast connectivity, all-day endurance, and a design you’ll want to show off.\n\nKey Features at a Glance\n- 6.5-inch OLED display: Immersive visuals with vibrant color, deep blacks, and smooth motion for gaming, movies, and everything in between.\n- Triple-lens camera system: Versatile setups for ultra-wide, wide, and telephoto shots, delivering crisp detail and flexible shooting options in any light.\n- 5G connectivity: Seamless streaming, faster downloads, and responsive video calls with ultra-low latency.\n- Long-lasting battery life: All-day power to keep you connected, entertained, and productive from morning to night.\n- Sleek design: A premium, slim profile with refined materials that feels great in your hand and looks stunning in any setting.\n\nWhy TechPhone X Stands Out\n- Brilliant visuals: The 6.5" OLED panel brings movies, photos, and games to life with e

In [5]:
# Web scraping and data extraction
import requests
from bs4 import BeautifulSoup

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

def fetch_website_content(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser") 
    title = soup.title.string if soup.title else "No title found"

    if soup.body:
        for irrelevant in soup.body(["script", "style", "nav", "footer"]):
            irrelevant.decompose()
            text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = "No body content found"

    return (title + "\n\n" + text)[:2_000]    

def fetch_website_links(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = []
    for a_tag in soup.find_all("a", href=True):
        href = a_tag["href"]
        if href.startswith("http"):
            links.append(href)
    return [link for link in links if link]
    
     


In [9]:
links = fetch_website_links("https://www.apple.com/iphone-14-pro/")
links

['https://support.apple.com/?cid=gn-ols-home-hp-tab',
 'https://support.apple.com/HT201196',
 'https://contactretail.apple.com/?pg=iphone&ap=COM&c=us&l=en&ag=SWSOV',
 'https://apps.apple.com/us/app/apple-store/id375380948/',
 'https://contactretail.apple.com/?pg=iphone&ap=COM&c=us&l=en&ag=SWSOV',
 'https://apps.apple.com/us/app/apple-store/id375380948/',
 'https://support.apple.com/121115?cid=general-iphone-marcom-08062025',
 'https://support.apple.com/121115?cid=general-iphone-marcom-08062025#visual-intelligence',
 'https://support.apple.com/121429?cid=general-iphone-marcom-08062025',
 'https://support.apple.com/kb/HT213885?cid=general-iphone-marcom-08062025',
 'https://www.goldmansachs.com/terms-and-conditions/Apple-Card-Customer-Agreement.pdf',
 'https://support.apple.com/102730?cid=general-phone-marcom-08062025',
 'https://www.goldmansachs.com/terms-and-conditions/Apple-Card-Customer-Agreement.pdf',
 'https://support.apple.com/102775?cid=general-iphone-marcom-08062025',
 'https://a

#### Building JSON Prompts and Using OpenAI's Chat Completions API

##### One shot prompting 

In [10]:
link_system_prompt = """ 
You are provided with a list of website links. 
You are able to decide which links are relevant to the task of generating a sales brochure the peoduct
such as links to product pages, reviews, and news articles about the product.
You should respond in JSON as this example:
{
    "links":[
    {"type": "product_page", "url": "https://www.example.com/product/techphone-x"},
    {"type": "review", "url": "https://www.example.com/review/techphone-x"},
    {"type": "news_article", "url": "https://www.example.com/news/techphone-x-launch"}
    ]
}
"""

In [11]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is a list of links from the website {url}:
plese decide which links are relevant to the task of generating a sales brochure for the product on the website and categorize them into product pages, reviews, and news articles.
respond with the relevant links in JSON format as specified in the system prompt.
Links (some might be relavant, some might not be):
"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [12]:
print(get_links_user_prompt("https://www.apple.com/iphone-14-pro/"))


Here is a list of links from the website https://www.apple.com/iphone-14-pro/:
plese decide which links are relevant to the task of generating a sales brochure for the product on the website and categorize them into product pages, reviews, and news articles.
respond with the relevant links in JSON format as specified in the system prompt.
Links (some might be relavant, some might not be):
https://support.apple.com/?cid=gn-ols-home-hp-tab
https://support.apple.com/HT201196
https://contactretail.apple.com/?pg=iphone&ap=COM&c=us&l=en&ag=SWSOV
https://apps.apple.com/us/app/apple-store/id375380948/
https://contactretail.apple.com/?pg=iphone&ap=COM&c=us&l=en&ag=SWSOV
https://apps.apple.com/us/app/apple-store/id375380948/
https://support.apple.com/121115?cid=general-iphone-marcom-08062025
https://support.apple.com/121115?cid=general-iphone-marcom-08062025#visual-intelligence
https://support.apple.com/121429?cid=general-iphone-marcom-08062025
https://support.apple.com/kb/HT213885?cid=general-

In [13]:
def select_relevent_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": link_system_prompt
            },
            {
                "role": "user",
                "content": get_links_user_prompt(url)
            },
        ],
        response_format={
                "type": "json_object"
            }
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [15]:
select_relevent_links("https://www.dialog.lk/")

{'links': [{'type': 'product_page', 'url': 'https://ideamart.lk/index.html'},
  {'type': 'news_article', 'url': 'https://www.facebook.com/dialog.lk/'},
  {'type': 'news_article', 'url': 'https://twitter.com/dialoglk'},
  {'type': 'news_article', 'url': 'https://www.instagram.com/dialoglk'},
  {'type': 'news_article', 'url': 'https://www.youtube.com/c/dialoglk'},
  {'type': 'news_article',
   'url': 'https://www.linkedin.com/company/dialog-axiata-plc/'}]}

## Make the brochure

In [ ]:
def fetech_page_and_all_relavent_links(url):
    links = select_relevent_links(url)
    content = {}
    for link in links["links"]:
        page_content = fetch_website_content(link["url"])
        content[link["url"]] = {
            "type": link["type"],
            "content": page_content
        }
    return content